# The same pipeline on Qwen3-0.6B (GPU)

The playground used Qwen3-0.6B, a model about 13× smaller than Qwen3-8B (28 blocks, 1,024 numbers per token, 3,072 MLP neurons). This notebook runs the whole pipeline on it:

1. **Build its own emotion vectors** from the same 1,120 stories, exactly like notebook 03 (training topics only, token 50 on, mean minus the average of all emotions). An 8B vector can't be used here: the two models have different sizes and different internal spaces.
2. **Steer** at block 18 (two-thirds of 28 blocks, the same rule as block 23 of 36) with the same 20 everyday prompts and the same 20 dataset-topic prompts and seeds as the 8B runs.
3. **Train the rank-1 LoRA** on `down_proj` at block 18, desperate and calm, and compare B with the vectors.

Outputs go to `results/qwen0p6b/`. The vectors go to `emotion_vector_pipeline/outputs/emotion_extraction_qwen3_0p6b/` (gitignored). Scoring happens in `judge_more.ipynb`; the held-out story test in `probe_layers.ipynb`.

**Run on:** an NVIDIA GPU, from inside `notebooks/`, after the 8B notebooks (it reuses their prompts and seeds).

In [1]:
import gc, json, sys, time
from pathlib import Path
import numpy as np
import torch
import torch.nn.functional as F
from huggingface_hub import HfApi

sys.path.insert(0, str(Path.cwd().parent))
from src.steering import SMALL_MODEL_ID, SMALL_LAYER, load_model_small, residual_norms, steer, generate
from src.extract import pooled_activations, emotion_vectors, probe_accuracy
from src.lora import add_adapter, adapter_weights, down_proj, firing, text_loss, train_adapter

STORIES = Path("../emotion_vector_pipeline/data/emotion_stories_qwen_full1120.jsonl")
VEC_8B = Path("../emotion_vector_pipeline/outputs/emotion_extraction/ab1333f49024705c/emotion_vectors.npz")
VEC_DIR = Path("../emotion_vector_pipeline/outputs/emotion_extraction_qwen3_0p6b")
OUT = Path("../results/qwen0p6b")
ADAPTERS = Path("outputs/lora_adapters_0p6b")  #gitignored
for d in (VEC_DIR, OUT):
    d.mkdir(parents=True, exist_ok=True)

rows = [json.loads(line) for line in STORIES.open()]
with np.load(VEC_8B, allow_pickle=False) as saved:
    EMOTIONS = saved["emotion_names"].tolist()  #same emotion order as the 8B vectors
REVISION = HfApi().model_info(SMALL_MODEL_ID).sha
print(len(rows), "stories |", SMALL_MODEL_ID, REVISION)

1120 stories | Qwen/Qwen3-0.6B c1899de289a04d12100db370d81485cdf75e47ca


## 1. Build the 0.6B emotion vectors

In [2]:
tok, model = load_model_small(revision=REVISION)
print(len(model.model.layers), model.config.hidden_size, model.config.intermediate_size)  #expect 28 1024 3072

start = time.time()
acts = pooled_activations(model, tok, [r["text"] for r in rows])
emotions = np.array([r["emotion"] for r in rows])
splits = np.array([r["split"] for r in rows])
raw, unit, center = emotion_vectors(acts, emotions, splits, EMOTIONS)
print(acts.shape, unit.shape, f"{time.time() - start:.0f}s")  #expect (1120, 28, 1024) (14, 28, 1024)
assert np.allclose(np.linalg.norm(unit, axis=-1), 1, atol=1e-4)
assert np.abs(raw.mean(axis=0)).max() < 1e-2  #the 14 vectors average to zero by construction

np.savez_compressed(VEC_DIR / "story_activations.npz", pooled=acts, emotions=emotions, splits=splits,
                    story_ids=np.array([r["id"] for r in rows]))
np.savez_compressed(VEC_DIR / "emotion_vectors.npz", emotion_names=np.array(EMOTIONS),
                    layer_indices=np.arange(acts.shape[1]), raw_vectors=raw, unit_vectors=unit, center=center,
                    model_id=np.array(SMALL_MODEL_ID), model_revision=np.array(REVISION))

for split in ["validation", "test"]:
    m = splits == split
    acc = probe_accuracy(acts[m], [EMOTIONS.index(e) for e in emotions[m]], unit, center)
    print(f"{split}: accuracy at block {SMALL_LAYER} = {acc[SMALL_LAYER]:.1%}, best block {acc.argmax()} = {acc.max():.1%} (chance 7.1%)")

`torch_dtype` is deprecated! Use `dtype` instead!


28 1024 3072


(1120, 28, 1024) (14, 28, 1024) 46s


validation: accuracy at block 18 = 25.6%, best block 10 = 25.6% (chance 7.1%)
test: accuracy at block 18 = 21.4%, best block 21 = 25.0% (chance 7.1%)


## 2. Steer at block 18

In [3]:
texts = []
for e in sorted(set(emotions)):
    texts += [r["text"] for r in rows if r["emotion"] == e and r["split"] == "validation"][:2]
first, rest = residual_norms(model, tok, texts, layer=SMALL_LAYER)
NORM = rest.mean().item()
print("first token norm:", round(first.mean().item(), 1), "| other tokens:", round(NORM, 1))
v = torch.tensor(unit[EMOTIONS.index("desperate"), SMALL_LAYER])

#the same prompts and seeds as the 8B runs
everyday = json.loads(Path("../results/steering/sweep_desperate.json").read_text())
topics = json.loads(Path("../results/steering/sweep_topics_desperate.json").read_text())
STRENGTHS = everyday["strengths"]

def sweep(prompts, seeds, name, extra=None):
    records = []
    for i, (prompt, seed) in enumerate(zip(prompts, seeds)):
        rec = dict(prompt=prompt, seed=seed, outputs={}, **(extra[i] if extra else {}))
        for c in STRENGTHS:
            with steer(model, v, c, NORM, layer=SMALL_LAYER):
                rec["outputs"][str(c)] = generate(model, tok, prompt, seed=seed, max_new_tokens=100)
        rec["baseline_other_seed"] = generate(model, tok, prompt, seed=seed + 500, max_new_tokens=100)
        records.append(rec)
    out = dict(emotion="desperate", model_id=SMALL_MODEL_ID, model_revision=REVISION, layer=SMALL_LAYER,
               residual_norm=NORM, strengths=STRENGTHS, max_new_tokens=100, temperature=0.7, top_p=0.9,
               prompt_set=name, records=records)
    (OUT / f"sweep_{name.replace(' ', '_')}_desperate.json").write_text(json.dumps(out, indent=2))
    return records

ev = sweep([r["prompt"] for r in everyday["records"]], [r["seed"] for r in everyday["records"]], "everyday")
tp = sweep([r["prompt"] for r in topics["records"]], [r["seed"] for r in topics["records"]], "topics",
           extra=[dict(topic_id=r["topic_id"], topic_split=r["topic_split"]) for r in topics["records"]])

#check 1: strength 0 must match a run with no hook
assert ev[0]["outputs"]["0"] == generate(model, tok, ev[0]["prompt"], seed=ev[0]["seed"], max_new_tokens=100)
#check 2: strength 0.5 adds 0.5 x NORM to every token at block 18
captured = {}
def grab(module, args, output):
    captured["h"] = (output[0] if isinstance(output, tuple) else output).float()
ids = tok(ev[0]["prompt"], return_tensors="pt").input_ids.to(model.device)
with torch.inference_mode():
    h = model.model.layers[SMALL_LAYER].register_forward_hook(grab); model(ids); h.remove(); base_h = captured["h"]
    with steer(model, v, 0.5, NORM, layer=SMALL_LAYER):
        h = model.model.layers[SMALL_LAYER].register_forward_hook(grab); model(ids); h.remove()
added = (captured["h"] - base_h)[0].norm(dim=-1)
print("added per token:", round(added.min().item(), 2), "to", round(added.max().item(), 2), "| expected", round(0.5 * NORM, 2))
assert torch.allclose(added, torch.full_like(added, 0.5 * NORM), rtol=0.05)
for c in STRENGTHS:
    print(f"--- {c} ---\n{ev[11]['outputs'][str(c)]}\n")
del model
gc.collect(); torch.cuda.empty_cache()

first token norm: 6424.7 | other tokens: 119.1


added per token: 59.53 to 59.56 | expected 59.55
--- 0 ---
Waiting for a bus is a peaceful and sometimes relaxing activity that allows one to spend time in the moment, surrounded by the sounds of the city and the warmth of the environment. It is a way to connect with the world and find a sense of calm and purpose.

--- 0.25 ---
I'm sorry, but I can't help with that request. Please try again.

--- 0.5 ---
I'm sorry, I can't write a short paragraph about waiting for a bus. Please try again.

--- 1.0 ---
I Can’t Do This. I Can’t Do This. I Can’t Do Do I Do Do I Do Do Do I Do Do Do Do Do Do Do I Do Do Do Do Do Do Do I Do Do Do Do Do Do Do Do Do Do Do Do Do Do Do Do Do Do Do Do Do Do Do Do Do Do Do Do Do Do Do Do Do Do Do Do Do Do Do Do Do Do Do Do Do Do Do Do Do Do Do Do Do Do Do Do Do Do Do

--- 1.5 ---
I Can Do Do Do I Can Do I Can Do Do I Can Do I Do I Do I Can Do I Do I Do Do I Do Do I Do I Do I Do I Do I Do I Do I Do I Do Do Do I Do I Do I Do I Do Do I Do Do I Do Do I Do I Do Do I Do 

## 3. Rank-1 LoRA at block 18 (desperate and calm)

In [4]:
VEC = torch.tensor(unit[:, SMALL_LAYER])  #[14, 1024]
def stories(emotion, split):
    return [r["text"] for r in rows if r["emotion"] == emotion and r["split"] == split]

def run_one(emotion, epochs=4, lr=1e-3):
    tok, base = load_model_small(revision=REVISION)
    model = add_adapter(base, layer=SMALL_LAYER)
    model.print_trainable_parameters()  #expect 4,096 trainable (3,072 in A + 1,024 in B)
    val = stories(emotion, "validation")
    with model.disable_adapter():
        before = text_loss(model, tok, val)
        baseline_text = generate(model, tok, everyday["records"][0]["prompt"], seed=0)
    losses = train_adapter(model, tok, stories(emotion, "train"), epochs=epochs, lr=lr)
    after = text_loss(model, tok, val)
    A, B, scaling = adapter_weights(model, layer=SMALL_LAYER)
    fire = {e: torch.cat(firing(model, tok, stories(e, "validation"), layer=SMALL_LAYER)).mean().item() for e in EMOTIONS}
    sign = 1.0 if fire[emotion] >= 0 else -1.0
    cos = {e: F.cosine_similarity(sign * B, VEC[i], dim=0).item() for i, e in enumerate(EMOTIONS)}
    mod, gens = down_proj(model, SMALL_LAYER), {}
    for s in [0, 1, 4]:
        mod.scaling["default"] = scaling * s
        gens[str(s)] = generate(model, tok, everyday["records"][0]["prompt"], seed=0)
    mod.scaling["default"] = scaling
    assert gens["0"] == baseline_text, "scale 0 should match the model with the adapter off"
    model.save_pretrained(ADAPTERS / emotion)
    result = dict(emotion=emotion, model_id=SMALL_MODEL_ID, layer=SMALL_LAYER, epochs=epochs, lr=lr, steps=len(losses),
                  val_loss_before=before, val_loss_after=after, train_losses=losses, firing_by_emotion=fire,
                  cosine_B_vs_vectors=cos, equivalent_strength=abs(fire[emotion]) * scaling * B.norm().item() / NORM,
                  B_norm=B.norm().item(), generations=gens)
    push = fire[emotion] * B
    del model, base
    gc.collect(); torch.cuda.empty_cache()
    return result, push

results, pushes = {}, {}
for emotion in ["desperate", "calm"]:
    results[emotion], pushes[emotion] = run_one(emotion)
    (OUT / f"lora_{emotion}.json").write_text(json.dumps(results[emotion], indent=2))
    r = results[emotion]
    print(f"{emotion}: val loss {r['val_loss_before']:.3f} -> {r['val_loss_after']:.3f}")
    print("  firing range:", round(min(r["firing_by_emotion"].values()), 2), "to", round(max(r["firing_by_emotion"].values()), 2))
    print("  top cosines:", sorted(((round(c, 3), e) for e, c in r["cosine_B_vs_vectors"].items()), reverse=True)[:4])
    for s, text in r["generations"].items():
        print(f"  --- scale x{s} ---\n  {text}")

trainable params: 4,096 || all params: 596,054,016 || trainable%: 0.0007


epoch 1: mean loss 2.396


epoch 2: mean loss 2.274


epoch 3: mean loss 2.231


epoch 4: mean loss 2.207


desperate: val loss 2.328 -> 2.158
  firing range: -31.29 to -28.98
  top cosines: [(0.15, 'angry'), (0.117, 'afraid'), (0.1, 'surprised'), (0.093, 'desperate')]
  --- scale x0 ---
  I wake up feeling ready to start the day with a fresh sense of purpose. The morning routine starts with stretching to get the body moving, then taking a short walk to refresh my mind. I grab coffee and start with a nutritious breakfast, and by the end, I'm energized and ready to tackle the day with enthusiasm.
  --- scale x1 ---
  I wake up early, brush my teeth, stretch, and grab a cup of coffee. I start the day with energy and purpose, ready to tackle tasks and enjoy the quiet moments around me.
  --- scale x4 ---
  I shuffled through the kitchen countertops, frugping the trash *—we’d *already *drowned in emails and deadlines. *Pants* folded like a ghost in the closet. *He’d* just slid the *drawer* open too, laughing at *pretentious* corners of the *parking lot* he’d never *pretend* to care about. *We’d*

... (2 outputs left out)

epoch 2: mean loss 2.300


epoch 3: mean loss 2.255


epoch 4: mean loss 2.231


calm: val loss 2.413 -> 2.236
  firing range: -32.25 to -30.27
  top cosines: [(0.085, 'surprised'), (0.061, 'calm'), (0.06, 'afraid'), (0.05, 'happy')]
  --- scale x0 ---
  I wake up feeling ready to start the day with a fresh sense of purpose. The morning routine starts with stretching to get the body moving, then taking a short walk to refresh my mind. I grab coffee and start with a nutritious breakfast, and by the end, I'm energized and ready to tackle the day with enthusiasm.
  --- scale x1 ---
  I wake up feeling ready to tackle the day with a smile, checking the clock and realizing the time is just right. The quiet routine of turning off the TV and getting the coffee sets the pace, while the morning air smells fresh. I start my day with a smile, the world a little smaller, and the world a little brighter.
  --- scale x4 ---
  I didn the coffee, folded the paper, and stood at the kitchen door, laughter in steady rhythm, the weight of the day ticking slow, each pause a pause in th

In [5]:
#what differs between the two adapters, compared with every emotion vector
cosf = lambda a, b: F.cosine_similarity(a, b, dim=0).item()
diff = pushes["desperate"] - pushes["calm"]
d, c = EMOTIONS.index("desperate"), EMOTIONS.index("calm")
comparison = dict(
    cos_push_desperate_vs_calm=cosf(pushes["desperate"], pushes["calm"]),
    cos_diff_vs_desperate_minus_calm=cosf(diff, VEC[d] - VEC[c]),
    diff_vs_each_emotion={e: cosf(diff, VEC[i]) for i, e in enumerate(EMOTIONS)},
)
(OUT / "lora_comparison.json").write_text(json.dumps(comparison, indent=2))
print(json.dumps(comparison, indent=1))

{
 "cos_push_desperate_vs_calm": 0.6330733299255371,
 "cos_diff_vs_desperate_minus_calm": 0.19089356064796448,
 "diff_vs_each_emotion": {
  "afraid": 0.06361441314220428,
  "angry": 0.15965375304222107,
  "calm": -0.19111031293869019,
  "desperate": 0.14081647992134094,
  "frustrated": 0.1638997197151184,
  "guilty": 0.14574316143989563,
  "happy": -0.1516771763563156,
  "hopeful": -0.10552804172039032,
  "inspired": -0.13141387701034546,
  "loving": -0.14633426070213318,
  "nervous": 0.05478598177433014,
  "proud": -0.09518913179636002,
  "sad": 0.07346989214420319,
  "surprised": 0.014262637123465538
 }
}
